# Chapter 15 - Monitoring Unpaid Claim Estimates

> Deviations of actual development from projected development of claims or
> claim counts are one of the most useful diagnostic tools for evaluating the
> accuracy of the unpaid claim estimate.
>
> -- Friedland, Chapter 15

The last part of Chapter 15 is not another projection method. It is a
**roll-forward**: take the ultimates and reporting pattern selected at one
valuation, and compare actual reported claims in the next period with the
amount that pattern said should emerge.

This notebook recreates Friedland's **DC Insurer** monitoring exhibits
(*Exhibit IV, Sheets 2–4*). The quarterly development triangle in Sheet 1 is
not shipped as a sample — the printed latest diagonals and selected CDFs are
enough to run the actual-versus-expected tests.

For each accident year, expected reported claims between two valuation dates
are

$$
\frac{\text{Ultimate}_{t_0} - \text{Reported}_{t_0}}{1 - p_{t_0}}
\times (p_{t_1} - p_{t_0})
$$

where $p_t$ is the selected percent reported at time $t$, equal to $1 / \text{CDF}_t$.

In [1]:
import numpy as np
import pandas as pd
import chainladder as cl
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)


def expected_reported(ultimate, reported, pct_from, pct_to):
    """Expected emergence between two valuations (Friedland Exhibit IV)."""
    unreported = 1.0 - pct_from
    emergence = np.where(unreported > 0, (ultimate - reported) / unreported, 0.0)
    return emergence * (pct_to - pct_from)

## Exhibit IV, Sheet 2 — Ultimates at 12/31/2007

DC Insurer selects ultimates with the reported development technique. The
12/31/2007 latest diagonal and the selected CDFs to ultimate are taken from
the printed exhibit (values in $000). Age 12 uses a 1.136 CDF (88.0%
reported); age 24 uses 1.001 (99.9% reported); older years are at 1.000.

`DevelopmentConstant` attaches those CDFs and `Chainladder` produces the
Sheet 2 ultimates. The text's worked example for accident year 2007 is
$2{,}463 \times 1.136 = 2{,}798$.

In [2]:
years = np.arange(1997, 2008)
reported_2007 = np.array(
    [3376, 2788, 1649, 1687, 2088, 2355, 2994, 3412, 2814, 2949, 2463],
    dtype=float,
)
ages = (2007 - years) * 12 + 12
cdf_by_age = {12: 1.136, 24: 1.001, **{age: 1.000 for age in ages if age >= 36}}

snapshot = pd.DataFrame(
    {
        "Accident Year": years,
        "Calendar Year": 2007,
        "Reported Claims": reported_2007,
    }
)
tri = cl.Triangle(
    snapshot,
    origin="Accident Year",
    development="Calendar Year",
    columns="Reported Claims",
    cumulative=True,
)
dev = cl.DevelopmentConstant(patterns=cdf_by_age, style="cdf").fit_transform(tri)
cl_model = cl.Chainladder().fit(dev)

sheet2 = pd.DataFrame(index=years)
sheet2["Age"] = ages
sheet2["Reported at 12/31/07"] = reported_2007
sheet2["CDF to Ultimate"] = [cdf_by_age[age] for age in ages]
sheet2["Projected Ultimate"] = np.round(
    cl_model.ultimate_.to_frame(origin_as_datetime=False).iloc[:, 0].values, 0
)
display(sheet2)
print(f"Total projected ultimate: {sheet2['Projected Ultimate'].sum():,.0f}")

,Age,Reported at 12/31/07,CDF to Ultimate,Projected Ultimate
1997,132,3376.0,1.000,3376.0
1998,120,2788.0,1.000,2788.0
1999,108,1649.0,1.000,1649.0
2000,96,1687.0,1.000,1687.0
2001,84,2088.0,1.000,2088.0
2002,72,2355.0,1.000,2355.0
2003,60,2994.0,1.000,2994.0
2004,48,3412.0,1.000,3412.0
2005,36,2814.0,1.000,2814.0
2006,24,2949.0,1.001,2952.0


Total projected ultimate: 28,913


## Exhibit IV, Sheet 3 — Annual monitoring test

One year later, compare calendar-year 2008 actual reported claims with the
amount implied by the 12/31/2007 ultimates and reporting pattern. The text
works accident year 2007 as

$$
\frac{2{,}798 - 2{,}463}{1 - 0.880} \times (0.999 - 0.880) = 332
$$

and accident year 2006 as

$$
\frac{2{,}952 - 2{,}949}{1 - 0.999} \times (1.000 - 0.999) = 3.
$$

Older years are fully reported, so expected emergence is zero.

In [3]:
pct_2007 = 1.0 / sheet2["CDF to Ultimate"].to_numpy()
# One year later each origin is 12 months older. Age 12 -> 24 (99.9%),
# age 24 -> 36 (100%), and mature years stay at 100%.
cdf_2008 = np.array([cdf_by_age.get(age + 12, 1.000) for age in ages])
pct_2008 = 1.0 / cdf_2008

reported_2008 = np.array(
    [3376, 2788, 1649, 1687, 2096, 2340, 3007, 3392, 2885, 3030, 2733],
    dtype=float,
)
ultimate = sheet2["Projected Ultimate"].to_numpy()
expected = np.round(expected_reported(ultimate, reported_2007, pct_2007, pct_2008), 0)
actual = reported_2008 - reported_2007

sheet3 = pd.DataFrame(index=years)
sheet3["Selected Ultimate"] = ultimate
sheet3["% Reported 12/31/07"] = np.round(pct_2007, 3)
sheet3["% Reported 12/31/08"] = np.round(pct_2008, 3)
sheet3["Reported 12/31/07"] = reported_2007
sheet3["Reported 12/31/08"] = reported_2008
sheet3["Actual"] = actual
sheet3["Expected"] = expected
sheet3["Difference"] = actual - expected
display(sheet3)
display(sheet3[["Actual", "Expected", "Difference"]].sum().rename("Total").to_frame().T)

C:\Users\EthanKang\AppData\Local\Temp\ipykernel_21808\827484566.py:13: RuntimeWarning: invalid value encountered in divide
  emergence = np.where(unreported > 0, (ultimate - reported) / unreported, 0.0)


,Selected Ultimate,% Reported 12/31/07,% Reported 12/31/08,Reported 12/31/07,Reported 12/31/08,Actual,Expected,Difference
1997,3376.0,1.000,1.000,3376.0,3376.0,0.0,0.0,0.0
1998,2788.0,1.000,1.000,2788.0,2788.0,0.0,0.0,0.0
1999,1649.0,1.000,1.000,1649.0,1649.0,0.0,0.0,0.0
2000,1687.0,1.000,1.000,1687.0,1687.0,0.0,0.0,0.0
2001,2088.0,1.000,1.000,2088.0,2096.0,8.0,0.0,8.0
2002,2355.0,1.000,1.000,2355.0,2340.0,-15.0,0.0,-15.0
2003,2994.0,1.000,1.000,2994.0,3007.0,13.0,0.0,13.0
2004,3412.0,1.000,1.000,3412.0,3392.0,-20.0,0.0,-20.0
2005,2814.0,1.000,1.000,2814.0,2885.0,71.0,0.0,71.0
2006,2952.0,0.999,1.000,2949.0,3030.0,81.0,3.0,78.0


,Actual,Expected,Difference
Total,408.0,335.0,73.0


## Exhibit IV, Sheet 4 — Monthly monitoring test

DC Insurer has quarterly development factors. Monthly percent-reported values
are **linear interpolations of the quarterly percent reported**. Between age 12
(88.0%) and age 15 ($1 / 1.016 \approx 98.4%$) that gives 91.5% at 13 months
and 95.0% at 14 months, matching the printed January / February 2008 template.

The same actual-versus-expected formula is applied month by month.

In [4]:
quarterly_cdf = {12: 1.136, 15: 1.016, 24: 1.001, 36: 1.000}


def pct_reported_at(age):
    """Linearly interpolate percent reported between quarterly CDF ages."""
    knots = np.array(sorted(quarterly_cdf))
    pcts = 1.0 / np.array([quarterly_cdf[k] for k in knots])
    if age <= knots[0]:
        return float(pcts[0])
    if age >= knots[-1]:
        return 1.0
    return float(np.interp(age, knots, pcts))


pct_jan = np.array([pct_reported_at(age + 1) for age in ages])
pct_feb = np.array([pct_reported_at(age + 2) for age in ages])

# Printed latest reported at 1/31/08 and 2/29/08 for the two immature years;
# mature years have no expected emergence, so the January/February actuals
# are taken from the printed Sheet 4 template where they are non-zero.
reported_jan = reported_2007.copy()
reported_feb = reported_2007.copy()
reported_jan[-1] = 2473  # AY 2007
reported_feb[-1] = 2538
reported_jan[-2] = 2951  # AY 2006
reported_feb[-2] = 2986

expected_jan = np.round(
    expected_reported(ultimate, reported_2007, pct_2007, pct_jan), 0
)
expected_feb = np.round(
    expected_reported(ultimate, reported_jan, pct_jan, pct_feb), 0
)
actual_jan = reported_jan - reported_2007
actual_feb = reported_feb - reported_jan

sheet4 = pd.DataFrame(index=years)
sheet4["Selected Ultimate"] = ultimate
sheet4["% Reported 12/31/07"] = np.round(pct_2007, 3)
sheet4["% Reported 1/31/08"] = np.round(pct_jan, 3)
sheet4["% Reported 2/29/08"] = np.round(pct_feb, 3)
sheet4["Reported 12/31/07"] = reported_2007
sheet4["Reported 1/31/08"] = reported_jan
sheet4["Reported 2/29/08"] = reported_feb
sheet4["Actual Jan"] = actual_jan
sheet4["Expected Jan"] = expected_jan
sheet4["Diff Jan"] = actual_jan - expected_jan
sheet4["Actual Feb"] = actual_feb
sheet4["Expected Feb"] = expected_feb
sheet4["Diff Feb"] = actual_feb - expected_feb
display(sheet4)
display(
    sheet4[["Actual Jan", "Expected Jan", "Diff Jan", "Actual Feb", "Expected Feb", "Diff Feb"]]
    .sum()
    .rename("Total")
    .to_frame()
    .T
)

C:\Users\EthanKang\AppData\Local\Temp\ipykernel_21808\827484566.py:13: RuntimeWarning: invalid value encountered in divide
  emergence = np.where(unreported > 0, (ultimate - reported) / unreported, 0.0)


,Selected Ultimate,% Reported 12/31/07,% Reported 1/31/08,% Reported 2/29/08,Reported 12/31/07,Reported 1/31/08,Reported 2/29/08,Actual Jan,Expected Jan,Diff Jan,Actual Feb,Expected Feb,Diff Feb
1997,3376.0,1.000,1.000,1.000,3376.0,3376.0,3376.0,0.0,0.0,0.0,0.0,0.0,0.0
1998,2788.0,1.000,1.000,1.000,2788.0,2788.0,2788.0,0.0,0.0,0.0,0.0,0.0,0.0
1999,1649.0,1.000,1.000,1.000,1649.0,1649.0,1649.0,0.0,0.0,0.0,0.0,0.0,0.0
2000,1687.0,1.000,1.000,1.000,1687.0,1687.0,1687.0,0.0,0.0,0.0,0.0,0.0,0.0
2001,2088.0,1.000,1.000,1.000,2088.0,2088.0,2088.0,0.0,0.0,0.0,0.0,0.0,0.0
2002,2355.0,1.000,1.000,1.000,2355.0,2355.0,2355.0,0.0,0.0,0.0,0.0,0.0,0.0
2003,2994.0,1.000,1.000,1.000,2994.0,2994.0,2994.0,0.0,0.0,0.0,0.0,0.0,0.0
2004,3412.0,1.000,1.000,1.000,3412.0,3412.0,3412.0,0.0,0.0,0.0,0.0,0.0,0.0
2005,2814.0,1.000,1.000,1.000,2814.0,2814.0,2814.0,0.0,0.0,0.0,0.0,0.0,0.0
2006,2952.0,0.999,0.999,0.999,2949.0,2951.0,2986.0,2.0,0.0,2.0,35.0,0.0,35.0


,Actual Jan,Expected Jan,Diff Jan,Actual Feb,Expected Feb,Diff Feb
Total,12.0,97.0,-85.0,100.0,132.0,-32.0


### Reconciliation to Friedland

The two worked examples and the interpolated January / February 2008 percents
are reconciled to the printed exhibit. Mature-year actuals on Sheet 3 come from
the printed 12/31/2008 diagonal and are not re-derived.

In [5]:
# Exhibit IV, Sheet 2 — AY 2007 and 2006 ultimates
assert sheet2.loc[2007, "Projected Ultimate"] == 2798
assert sheet2.loc[2006, "Projected Ultimate"] == 2952
assert sheet2.loc[2007, "CDF to Ultimate"] == 1.136
assert sheet2.loc[2006, "CDF to Ultimate"] == 1.001

# Exhibit IV, Sheet 3 — worked examples and total expected emergence
assert sheet3.loc[2007, "% Reported 12/31/07"] == 0.880
assert sheet3.loc[2007, "% Reported 12/31/08"] == 0.999
assert sheet3.loc[2007, "Expected"] == 332
assert sheet3.loc[2006, "Expected"] == 3
assert sheet3["Expected"].sum() == 335

# Exhibit IV, Sheet 4 — interpolated percent reported for AY 2007
assert np.isclose(sheet4.loc[2007, "% Reported 1/31/08"], 0.915, atol=5e-4)
assert np.isclose(sheet4.loc[2007, "% Reported 2/29/08"], 0.950, atol=5e-4)